# [9-1강] seed 고정과 재현성 요소 - 실습

In [1]:
import random
import json
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split

# 실습 결과가 매번 비슷하게 나오도록 seed를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


## 필수 1 1: seed 고정 함수 작성하기

### 문제 설명
실험 시작 전에 호출할 set_seed 함수를 만듭니다.

In [2]:
def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
    # TODO: random, numpy, torch seed를 모두 고정하세요.

set_seed(123)
a = torch.randn(3)
set_seed(123)
b = torch.randn(3)
print('a:', a)
print('b:', b)
print('same:', torch.equal(a, b))

a: tensor([-0.1115,  0.1204, -0.3696])
b: tensor([-0.1115,  0.1204, -0.3696])
same: True


### 해설 및 실행 결과 해석
같은 seed에서 같은 Tensor가 나오면 난수 생성 흐름이 재현됩니다. 다만 GPU 연산이나 외부 라이브러리 설정에 따라 완전한 재현성은 별도 점검이 필요합니다.

## 필수 2 2: 같은 seed로 모델 초기화 비교하기

### 문제 설명
모델 생성 전에 seed를 고정하면 같은 초기 weight를 얻을 수 있습니다.

In [3]:
def make_model_with_seed(seed):
    # TODO: 모델 생성 전에 torch seed를 고정하세요.
    torch.manual_seed(seed)
    model = nn.Linear(2, 1)
    return model

m1 = make_model_with_seed(7)
m2 = make_model_with_seed(7)
print('same_weight:', torch.equal(m1.weight, m2.weight))

same_weight: True


### 해설 및 실행 결과 해석
seed는 모델 초기화에도 영향을 줍니다. baseline 실험을 비교하려면 데이터 분할뿐 아니라 모델 초기화 seed도 함께 기록해야 합니다.